In [1]:
#Install/Import libraries

import os
import numpy as np
import pandas as pd
import cv2
import glob

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Flatten, Concatenate, Input, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [4]:
# Clone the public "Houses Dataset" (images + tabular info combined)
!git clone https://github.com/emanhamed/Houses-dataset.git

DATASET_PATH = "/content/Houses-dataset/Houses Dataset"
print(os.listdir(DATASET_PATH)[:10])

fatal: destination path 'Houses-dataset' already exists and is not an empty directory.
['439_bedroom.jpg', '48_kitchen.jpg', '356_frontal.jpg', '124_bathroom.jpg', '282_frontal.jpg', '526_kitchen.jpg', '77_bathroom.jpg', '231_kitchen.jpg', '449_bathroom.jpg', '330_frontal.jpg']


In [5]:
# HousesInfo.txt has: bedrooms, bathrooms, area, zipcode, price (space-separated, no header)
columns = ["bedrooms", "bathrooms", "area", "zipcode", "price"]
tabular_df = pd.read_csv(
    os.path.join(DATASET_PATH, "HousesInfo.txt"),
    sep=" ",
    header=None,
    names=columns,
)

print("Shape:", tabular_df.shape)
tabular_df.head()

Shape: (535, 5)


,bedrooms,bathrooms,area,zipcode,price
0,4,4.0,4053,85255,869500
1,4,3.0,3343,36372,865200
2,3,4.0,3923,85266,889000
3,5,5.0,4022,85262,910000
4,3,4.0,4116,85266,971226


In [6]:
def load_house_images(df, path):
    """
    Each house has 4 images: bathroom, bedroom, frontal, kitchen.
    We combine them into a single 2x2 collage image so the CNN
    sees all 4 rooms of a house at once.
    """
    images = []
    for i in df.index.values:
        base_path = os.path.sep.join([path, f"{i + 1}_*"])
        house_paths = sorted(glob.glob(base_path))

        input_images = []
        output_image = np.zeros((64, 64, 3), dtype="uint8")

        for house_path in house_paths:
            image = cv2.imread(house_path)
            image = cv2.resize(image, (32, 32))
            input_images.append(image)

        # Arrange the 4 images into a 2x2 grid (64x64 total)
        output_image[0:32, 0:32] = input_images[0]
        output_image[0:32, 32:64] = input_images[1]
        output_image[32:64, 32:64] = input_images[2]
        output_image[32:64, 0:32] = input_images[3]

        images.append(output_image)

    return np.array(images)


images = load_house_images(tabular_df, DATASET_PATH)
print("Images array shape:", images.shape)

# Normalize pixel values to 0-1 range
images = images / 255.0

Images array shape: (535, 64, 64, 3)


In [7]:
# Scale numeric tabular features to 0-1 range
tab_scaler = MinMaxScaler()
tabular_features = tab_scaler.fit_transform(
    tabular_df[["bedrooms", "bathrooms", "area"]]
)

# Scale the target (price) too — helps training stability
price_scaler = MinMaxScaler()
prices_scaled = price_scaler.fit_transform(tabular_df[["price"]])

print("Tabular features shape:", tabular_features.shape)

Tabular features shape: (535, 3)


In [9]:
#Train/test split (images + tabular + target together)

split = train_test_split(
    tabular_features, images, prices_scaled,
    test_size=0.2, random_state=42
)

(X_tab_train, X_tab_test,
 X_img_train, X_img_test,
 y_train, y_test) = split

print("Train images:", X_img_train.shape, "Test images:", X_img_test.shape)

Train images: (428, 64, 64, 3) Test images: (107, 64, 64, 3)


In [10]:
#Build the CNN branch (extracts features from images)

def build_cnn_branch(input_shape=(64, 64, 3)):
    inputs = Input(shape=input_shape)

    # Simple CNN feature extractor (small dataset -> keep it lightweight)
    x = tf.keras.layers.Conv2D(16, (3, 3), padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = Flatten()(x)
    x = Dense(16, activation="relu")(x)   # this is the "image feature vector"

    return Model(inputs, x)


cnn_branch = build_cnn_branch()

In [11]:
#Build the tabular (MLP) branch

def build_mlp_branch(input_dim):
    inputs = Input(shape=(input_dim,))
    x = Dense(8, activation="relu")(inputs)
    x = Dense(4, activation="relu")(x)   # this is the "tabular feature vector"
    return Model(inputs, x)


mlp_branch = build_mlp_branch(X_tab_train.shape[1])

In [12]:
#Fuse both branches (feature fusion) and add final regression head

# Concatenate the outputs of both branches (image features + tabular features)
combined = Concatenate()([cnn_branch.output, mlp_branch.output])

x = Dense(8, activation="relu")(combined)
x = Dropout(0.2)(x)
output = Dense(1, activation="linear")(x)   # regression output (predicted price)

model = Model(inputs=[cnn_branch.input, mlp_branch.input], outputs=output)

model.compile(optimizer=Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │         64 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 16,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 8, 8, 64)  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4096)      │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8)         │         32 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │     65,552 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 4)         │         36 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 20)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 8)         │        168 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 8)         │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1)         │          9 │ dropout[0][0]   

 Total params: 89,829 (350.89 KB)

 Trainable params: 89,605 (350.02 KB)

 Non-trainable params: 224 (896.00 B)

In [13]:
#Train the multimodal model

history = model.fit(
    x=[X_img_train, X_tab_train],
    y=y_train,
    validation_data=([X_img_test, X_tab_test], y_test),
    epochs=30,
    batch_size=8,
)

Epoch 1/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 0.0252 - mae: 0.0946 - val_loss: 0.0053 - val_mae: 0.0621
Epoch 2/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - loss: 0.0106 - mae: 0.0633 - val_loss: 0.0207 - val_mae: 0.1328
Epoch 3/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - loss: 0.0089 - mae: 0.0564 - val_loss: 0.0416 - val_mae: 0.1886
Epoch 4/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - loss: 0.0094 - mae: 0.0579 - val_loss: 0.0360 - val_mae: 0.1613
Epoch 5/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - loss: 0.0085 - mae: 0.0571 - val_loss: 0.0471 - val_mae: 0.1690
Epoch 6/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - loss: 0.0085 - mae: 0.0570 - val_loss: 0.0520 - val_mae: 0.1858
Epoch 7/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - loss: 0.0085 - mae: 0.0572 - val_loss: 0.0393 - val_mae: 0.1356
Epoch 8/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 0.0085 - mae: 0.0583 - val_loss: 0.0085 - val_mae: 0.0638
Epoch 9/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 71ms/step - loss: 0.008

In [14]:
# Get predictions (still in scaled 0-1 form)
preds_scaled = model.predict([X_img_test, X_tab_test])

# Convert predictions and true values back to real dollar prices
preds_actual = price_scaler.inverse_transform(preds_scaled)
y_test_actual = price_scaler.inverse_transform(y_test)

mae = mean_absolute_error(y_test_actual, preds_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, preds_actual))

print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
MAE:  $285,289.48
RMSE: $376,616.60


In [16]:
#Save the trained model


model.save("housing_multimodal_model.keras")
print("Model saved as housing_multimodal_model.keras")

Model saved as housing_multimodal_model.keras


In [19]:
print("Average price:", tabular_df["price"].mean())
print("Median price:", tabular_df["price"].median())
print("Min price:", tabular_df["price"].min())
print("Max price:", tabular_df["price"].max())

Average price: 589362.8112149533
Median price: 529000.0
Min price: 22000
Max price: 5858000
